# Prepare FINDR dataset from pyNeuroDAP outputs

This notebook builds a `.npz` dataset compatible with FINDR, using outputs produced by `analyze_sessions_forPlexon.py` in this repo. It follows the data format described in the FINDR README: `spikes`, `externalinputs`, `lengths`, `times`.

Reference: Brody-Lab/findr (`https://github.com/Brody-Lab/findr`).


In [4]:
from pathlib import Path
import os
import numpy as np
import h5py
import pandas as pd

# Local pyNeuroDAP helpers
import pyNeuroDAP as ndap
os.chdir("/Users/shunli/Projects/pyNeuroDAP")

# Paths: point to a specific analyzed session directory
# Example (adjust to your session):
SESSION_DIR = Path('Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822')
DATA_H5 = SESSION_DIR / 'data.h5'
SPIKES_H5 = SESSION_DIR / 'aligned_spikes.h5'

# Output .npz path for FINDR
FINDR_NPZ = SESSION_DIR / 'findr_dataset.npz'

print('Data file:', DATA_H5)
print('Spikes file:', SPIKES_H5)
print('Output npz:', FINDR_NPZ)


Data file: Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/data.h5
Spikes file: Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/aligned_spikes.h5
Output npz: Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/findr_dataset.npz


In [5]:
# Load data.h5: trial table, event times, metadata
trial_df = ndap.load_dataframe(DATA_H5, key='trial_table')
event_times = ndap.load_variables(DATA_H5, key='event_times')['event_times']
metadata = ndap.load_variables(DATA_H5, key='metadata')['metadata']

print('trial_df shape:', trial_df.shape)
print('event_times keys:', list(event_times.keys()))
print('metadata keys:', list(metadata.keys()))


DataFrame loaded from Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/data.h5
Variables loaded from Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/data.h5
Variables loaded from Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/data.h5
trial_df shape: (525, 18)
event_times keys: ['choice_lick_times', 'last_lick_times', 'second_lick_times', 'trial_start_times']
metadata keys: ['trial_conditions', 'bin_size', 'experiment_type', 'laser_duration', 'laser_onset', 'recording_location', 'subject_id', 'time_range', 'trial_range']


In [6]:
# Load aligned spikes lazily (h5py datasets) for a chosen alignment key
# Choose one alignment (options saved by analyze script): 'trial_start', 'choice_lick', 'second_lick', 'last_lick'
ALIGNMENT_KEY = 'trial_start'

aligned = ndap.load_aligned_spikes(SPIKES_H5, lazy=True)
assert ALIGNMENT_KEY in aligned, f"Alignment key {ALIGNMENT_KEY} not found. Available: {list(aligned.keys())}"

aligned_key = aligned[ALIGNMENT_KEY]
# aligned_key is a dict per-condition -> dict with 'rate', 'times', etc.
conditions = list(aligned_key.keys())
print('conditions:', conditions)

# Inspect shapes
for cond in conditions:
    rate = aligned_key[cond]['rate']  # h5py.Dataset (units, trials, timebins)
    print(cond, 'rate shape:', rate.shape)


Aligned spikes loaded from Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/aligned_spikes.h5 (lazy=True)
conditions: ['nonreward_left_control', 'nonreward_left_laser', 'nonreward_right_control', 'nonreward_right_laser', 'reward_left_control', 'reward_left_laser', 'reward_right_control', 'reward_right_laser']
nonreward_left_control rate shape: (142, 30, 120)
nonreward_left_laser rate shape: (142, 35, 120)
nonreward_right_control rate shape: (142, 83, 120)
nonreward_right_laser rate shape: (142, 42, 120)
reward_left_control rate shape: (142, 146, 120)
reward_left_laser rate shape: (142, 49, 120)
reward_right_control rate shape: (142, 103, 120)
reward_right_laser rate shape: (142, 37, 120)


In [8]:
# Build FINDR arrays
# FINDR expects:
# - spikes: (num_trials, max_T, num_neurons) of binned spike counts
# - externalinputs: (num_trials, max_T, input_dim)
# - lengths: (num_trials,) lengths per trial
# - times: (num_trials,) onset timestamp per trial

# We will:
# 1) pick a subset of conditions to include (or all)
# 2) concatenate trials across chosen conditions
# 3) transpose from (U,T,B) to (T,B,U) and pad to max_T per trial

use_conditions = conditions  # or subset like ['reward_right_laser', 'reward_left_laser']

# Time-varying inputs per bin: [tone_left, tone_right, laser_on]
input_dim = 3

# Helpers for time indexing
def parse_time_range(tr_val):
    if isinstance(tr_val, (list, tuple, np.ndarray)) and len(tr_val) >= 2:
        return float(tr_val[0]), float(tr_val[1])
    if isinstance(tr_val, str):
        s = tr_val.strip().strip('()[]')
        parts = [p for p in s.replace(',', ' ').split() if p]
        if len(parts) >= 2:
            return float(parts[0]), float(parts[1])
    # Fallback to (-1, 2) if unknown
    return -1.0, 2.0

bin_size_ms = float(metadata['bin_size'])
laser_onset_s = float(metadata.get('laser_onset', 1.0))
laser_dur_s = float(metadata.get('laser_duration', 0.5))
tr_start_s, tr_end_s = parse_time_range(metadata.get('time_range', (-1.0, 2.0)))

# Gather per-trial tensors
trial_spikes = []
trial_inputs = []
trial_lengths = []
trial_onsets = []

for cond in use_conditions:
    rate_ds = aligned_key[cond]['rate']        # shape: (units, trials, timebins)
    U, T, B = rate_ds.shape

    # Determine tone side from condition name
    is_left = 'left' in cond
    is_right = 'right' in cond
    has_laser = 'laser' in cond

    # Pre-compute time-to-bin conversion
    bin_size_s = bin_size_ms / 1000.0
    def sec_to_bin_index(t_sec):
        return int(np.clip(np.floor((t_sec - tr_start_s) / bin_size_s), 0, B - 1))

    tone_on_s, tone_off_s = 0.0, 0.025  # 25 ms tone aligned to trial start
    if tone_off_s <= tone_on_s:
        tone_off_s = tone_on_s

    for t in range(T):
        x = np.array(rate_ds[:, t, :], dtype=np.float32)  # (U,B)
        x = np.transpose(x, (1, 0))                       # (B,U)
        trial_spikes.append(x)

        # Build time-varying inputs per-bin
        B_local = x.shape[0]
        u = np.zeros((B_local, input_dim), dtype=np.float32)

        # Tone window
        i0 = sec_to_bin_index(tone_on_s)
        i1 = min(sec_to_bin_index(tone_off_s) + 1, B_local)
        if is_left and i0 < B_local:
            u[i0:i1, 0] = 1.0  # tone_left
        if is_right and i0 < B_local:
            u[i0:i1, 1] = 1.0  # tone_right

        # Laser window
        if has_laser:
            j0 = sec_to_bin_index(laser_onset_s)
            j1 = min(sec_to_bin_index(laser_onset_s + laser_dur_s) + 1, B_local)
            if j0 < B_local:
                u[j0:j1, 2] = 1.0  # laser_on

        trial_inputs.append(u)
        trial_lengths.append(B_local)

        # Trial onset time: look up from event_times['trial_start_times'][cond]
        onset_vec = event_times['trial_start_times'].get(cond, None)
        if onset_vec is not None and len(onset_vec) > t:
            trial_onsets.append(float(onset_vec[t]))
        else:
            trial_onsets.append(np.nan)

# Pad trials to max_T
max_T = max(trial_lengths) if trial_lengths else 0
num_trials = len(trial_spikes)
num_neurons = trial_spikes[0].shape[1] if num_trials > 0 else 0

spikes_arr = np.zeros((num_trials, max_T, num_neurons), dtype=np.float32)
inputs_arr = np.zeros((num_trials, max_T, input_dim), dtype=np.float32)
lengths_arr = np.array(trial_lengths, dtype=np.int32)
times_arr = np.array(trial_onsets, dtype=np.float64)

for i, (x, u) in enumerate(zip(trial_spikes, trial_inputs)):
    L = x.shape[0]
    spikes_arr[i, :L, :] = x
    inputs_arr[i, :L, :] = u

print('spikes:', spikes_arr.shape)
print('externalinputs:', inputs_arr.shape)
print('lengths:', lengths_arr.shape)
print('times:', times_arr.shape)


spikes: (525, 120, 142)
externalinputs: (525, 120, 3)
lengths: (525,)
times: (525,)


In [9]:
# Save .npz for FINDR
np.savez_compressed(
    FINDR_NPZ,
    spikes=spikes_arr,
    externalinputs=inputs_arr,
    lengths=lengths_arr,
    times=times_arr,
)
print('Saved:', FINDR_NPZ)


Saved: Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/findr_dataset.npz


## Run FINDR training
You can run FINDR as described in the repo README. Example command:

```bash
conda run -n pyNeuroDAP python /Users/shunli/Projects/pyNeuroDAP/findr/main.py \
  --datapath="${PWD}/Data/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/results-250822/findr_dataset.npz" \
  --workdir="${PWD}/Results/findr/Rec_Upstream_DCN_1_250411_MixedmW_500ms_041225001/250822"
```

Adjust paths as needed. It may take hours and expects JAX/Flax installed and working on your machine.

Reference: Brody-Lab/findr (`https://github.com/Brody-Lab/findr`).
